In [ ]:
from sqlite3 import connect
from os import path
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=x.dtype,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//6*16))
        s._ = Linear(o or i//6*16, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
class REnCoder(Sequential):
    def __init__(s, o, i, bin, *emb, drop=.1):
        super().__init__(Linear(i,o),EmbeddingBag(bin,o,scale_grad_by_freq=True,mode='sum'),*(Embedding(i,o,0,scale_grad_by_freq=True)for i in emb))
        s.drop = drop
    def forward(s, x):
        x = x.split((s[0].in_features,s[1].num_embeddings//64,*[1]*(len(s)-2)),-1)
        return sum(
            (s[i](x[...,0])for i,x in enumerate(x[2:],2)),
            s[0](functional.dropout(x[0].to(s[0].weight.dtype),s.drop,s.training,True))+
            s[1](
                (_:=(x[1][...,None]&1<<torch.arange(64,device=x[0].device)).flatten(0,-3).flatten(1).nonzero())[:,1],
                torch.zeros(1+x[0].shape[:-1].numel(),dtype=torch.int,device=x[0].device)[1:].scatter_add_(
                    0,
                    _[:,0],
                    torch.ones(1,dtype=torch.int,device=x[0].device).expand(len(_))
                )._base[:-1].cumsum(0)
            ).unflatten(0,x[0].shape[:-1])
        )
def pack(*a):
    x = 1
    for a in a:
        x <<= a[-1]
        for a in a[:-1]:
            x |= a if isinstance(a,int)else 1<<int(a)-1 if a else 0
    while x>1:
        yield x&0xffffffffffffffff
        x >>= 64
CARDS = {id:(
    max(atk,0),
    0 if type&0x4000000 else max(Def,0),
    level&15,
    level>>24,
    len(name),
    len(desc),
    len(_:=path.exists(_:=f'script/c{alias or id}.lua')and open(_).read()or''),
    *pack(
        (ot-(ot>0x4)*0x4,3),
        (*map(float,(setcode&0xfff,setcode>>16&0xfff,setcode>>32&0xfff,setcode>>48&0xfff)),475),
        ((setcode>>12|setcode>>28|setcode>>44|setcode>>60)&0xf,4),
        (type,27),
        (category%(1<<32),32),
        (Def-(Def>0x10)*0x10 if type&0x4000000 else 0,8),
        (sum((j in name)<<i for i,j in enumerate('- 星者魔生之战王光兽火灵剑人神骑女大天机士物子龙黑的·')),28),
        (sum((j in desc)<<i for i,j in enumerate('手把·攻「表方对时有不存个那果外为次墓在地坏殊从名特组用击破')),30),
        (sum((j in _)<<i for i,j in enumerate(
            'not GetLocationCount IsType IsLocation IsExistingMatchingCard IsFaceup tc target EFFECT_FLAG_CARD_TARGET false filter true POS_FACEUP SetValue LOCATION_GRAVE LOCATION_HAND REASON_EFFECT e3 EVENT_FREE_CHAIN '
            'SetReset EFFECT_TYPE_SINGLE EFFECT_TYPE_TRIGGER_O Card EFFECT_TYPE_ACTIVATE CATEGORY_SPECIAL_SUMMON SetCondition SetCost SetCountLimit SelectMatchingCard EFFECT_TYPE_FIELD GetCount IsRelateToEffect IsSetCard'.split()
        )),33),
    ),
    race.bit_length(),
    attribute.bit_length(),
    name,
    bool(type&0x4802040),
    alias or id,
)for id,ot,alias,setcode,type,atk,Def,level,race,attribute,category,name,desc in connect('cards.cdb').execute('select datas.*,name,desc from datas join texts on datas.id=texts.id')}
CARDS[0] = (0,)*22
LFS = dict(map(int,i.split()[:2])for i in open('lflist.conf').read().split('!')[1].split('\n')if'-'in i)

In [ ]:
name = lambda:'model5a'
MAP = dict(map(reversed,enumerate(MAP_:=np.fromfile(name()+'.i','i').tolist())))
CARD = torch.tensor([CARDS[i][:-3]for i in MAP],dtype=torch.uint64).view(torch.int64)
EX = torch.tensor([CARDS[i][-2]for i in MAP],dtype=torch.bool)
NEX = ~EX
LF = torch.tensor([LFS.get(i,3 if i else 255)for i in MAP],dtype=torch.uint8)
m = torch.load(name()+'.pk','cpu',weights_only=False)[0].requires_grad_(False).eval()

In [ ]:
x = torch.zeros(len(MAP),dtype=torch.uint8)
x[MAP[95365081]] = 3
ex = x@EX.byte()
for n in range(x.sum(),55):
    p = m(torch.concat((x[_:=x.nonzero()],CARD[_[:,0]],_),1))[-1]
    p[x==LF] = -9e9
    if ex==15:
        p[EX] = -9e9
    elif n-ex==40:
        p[NEX] = -9e9
    i = p.argmax()
    print(CARDS[MAP_[i]][-3])
    x[i] += 1

x = torch.zeros(5,len(MAP),dtype=torch.long)
x[0,MAP[23626223]] = 3
x[0,2] = 3
x[0,3] = 4
x[1,4] = 5
x[2,4] = 5
x[2,7] = 13
x[3,5] = 7
x[4,6] = 7
cump = torch.zeros(1)
ex = 15-x@EX
main = 55-x.sum(1)-ex
for _ in range((main+ex)[0]):
    t = torch.zeros(len(x),x.bool().sum(1).max(),dtype=torch.long)
    i,j = x.nonzero().T
    t[i,(_:=torch.arange(len(i))).sub_(i.diff(prepend=torch.tensor([-1])).mul_(_).cummax(0)[0])] = j
    p = m(torch.concat((x.gather(1,t)[:,:,None],CARD[t],t[:,:,None]),2))
    break